In [ ]:
import os
from typing import Annotated, Literal

import gradio as gr
from agents.mcp import MCPServerStdio
from dotenv import load_dotenv
from langchain_core.messages import AIMessage, HumanMessage, RemoveMessage, SystemMessage
from langchain_core.tools import StructuredTool
from langchain_openai import ChatOpenAI
from langgraph.graph import END, START, StateGraph
from langgraph.graph.message import REMOVE_ALL_MESSAGES, add_messages
from langgraph.prebuilt import ToolNode, tools_condition
from pydantic import BaseModel, Field
from typing_extensions import NotRequired, TypedDict

In [ ]:
load_dotenv(override=True)

In [ ]:

os.environ["PATH"] = "/usr/local/bin/node" + os.pathsep + os.environ["PATH"]
os.environ["PATH"] = "/usr/local/bin/npx" + os.pathsep + os.environ["PATH"]



In [ ]:
import subprocess
result = subprocess.run(["which", "node"], capture_output=True, text=True)
if result.returncode == 0:
    print(result.stdout.strip())
else:
    print(result.returncode, "missing")

print(os.environ["PATH"])


In [ ]:
sandbox_path = os.path.abspath(os.path.join(os.getcwd(), "sandbox"))
os.makedirs(sandbox_path, exist_ok=True)

playwright_params = {
    
    "command": "npx",
    "args": ["-y", "@playwright/mcp@latest"],
    "env": {  # Optional: pass environment variables if needed
        **os.environ,
    }
}


async with MCPServerStdio(params=playwright_params, client_session_timeout_seconds=60) as server:
    p_tools = await server.list_tools()

p_tools

In [ ]:
class State(TypedDict):
    messages: Annotated[list, add_messages]
    route: NotRequired[str]


llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.7)


def _format_mcp_tool_result(result) -> str:
    parts = []
    for block in getattr(result, "content", None) or []:
        text = getattr(block, "text", None)
        parts.append(text if text is not None else str(block))
    return "\n".join(parts) if parts else str(result)


class BrowserTabsArgs(BaseModel):
    action: Literal["list", "new", "close", "select"] = Field(
        ..., description='Tab operation: list open tabs, new tab, close tab, or select tab by index.'
    )
    index: int | None = Field(
        None, description="Tab index for close/select; omit for close to close current tab."
    )


class BrowserNavigateArgs(BaseModel):
    url: str = Field(..., description="Full URL to load in the current browser tab.")


class BrowserSnapshotArgs(BaseModel):
    filename: str | None = Field(
        None,
        description="Optional: save snapshot to this markdown file instead of returning inline.",
    )


async def build_navigator_tools(server) -> list:
    """Navigation agent: URLs + tabs (Playwright MCP)."""
    names = {t.name for t in await server.list_tools()}
    out = []
    if "browser_tabs" in names:

        async def _browser_tabs(action: str, index: int | None = None) -> str:
            payload = {"action": action}
            if index is not None:
                payload["index"] = index
            r = await server.call_tool("browser_tabs", payload)
            return _format_mcp_tool_result(r)

        out.append(
            StructuredTool.from_function(
                coroutine=_browser_tabs,
                name="browser_tabs",
                description="List, create (new), close, or select browser tabs by index.",
                args_schema=BrowserTabsArgs,
            )
        )
    if "browser_navigate" in names:

        async def _browser_navigate(url: str) -> str:
            r = await server.call_tool("browser_navigate", {"url": url})
            return _format_mcp_tool_result(r)

        out.append(
            StructuredTool.from_function(
                coroutine=_browser_navigate,
                name="browser_navigate",
                description="Open a URL in the current browser tab.",
                args_schema=BrowserNavigateArgs,
            )
        )
    return out


async def build_summarizer_tools(server) -> list:
    """Summarization agent: read page content via accessibility snapshot."""
    names = {t.name for t in await server.list_tools()}
    if "browser_snapshot" not in names:
        return []

    async def _browser_snapshot(filename: str | None = None) -> str:
        payload = {}
        if filename is not None:
            payload["filename"] = filename
        r = await server.call_tool("browser_snapshot", payload)
        return _format_mcp_tool_result(r)

    return [
        StructuredTool.from_function(
            coroutine=_browser_snapshot,
            name="browser_snapshot",
            description=(
                "Capture an accessibility snapshot of the current page (text structure). "
                "Call this before summarizing what the user sees in the browser."
            ),
            args_schema=BrowserSnapshotArgs,
        )
    ]


def compile_agent_subgraph(tools: list):
    """One agent = LLM (+ tools) with optional ToolNode loop (ReAct)."""
    if not tools:

        def agent_once(state: State) -> dict:
            return {"messages": [llm.invoke(state["messages"])]}

        b = StateGraph(State)
        b.add_node("agent", agent_once)
        b.add_edge(START, "agent")
        b.add_edge("agent", END)
        return b.compile()

    llm_tools = llm.bind_tools(tools)

    def agent_node(state: State) -> dict:
        return {"messages": [llm_tools.invoke(state["messages"])]}

    tool_node = ToolNode(tools)
    b = StateGraph(State)
    b.add_node("agent", agent_node)
    b.add_node("tools", tool_node)
    b.add_edge(START, "agent")
    b.add_conditional_edges("agent", tools_condition)
    b.add_edge("tools", "agent")
    return b.compile()


class AgentRoute(BaseModel):
    target: Literal["general", "navigator", "summarizer"] = Field(
        description="Which specialist should handle the latest user request."
    )


ROUTER_PROMPT = """You route each user turn to exactly one handler:
- **navigator**: open/go to URLs, manage tabs (list / new / close / switch), browser navigation only.
- **summarizer**: user wants a summary or explanation of the **current** web page ("what does this page say", etc.).
  The graph always runs the navigation agent first, then the snapshot/summary agent—pick **summarizer** for page-summary intent.
- **general**: anything else (no browser action needed).

Choose based on the latest user message (and short context)."""

AGENT_SYSTEM = {
    "general": (
        "You are a helpful assistant. You do not have browser tools; answer from general knowledge."
    ),
    "navigator": (
        "You are the Browser Navigation agent. Use only your tools (browser_navigate, browser_tabs) "
        "to open URLs and manage tabs. If the user will get a page summary next, ensure the correct URL/tab is active first. "
        "After your tools complete, reply briefly; a separate agent may summarize the page afterward."
    ),
    "summarizer": (
        "You are the Page Summarization agent. Use browser_snapshot to read the current page, then "
        "give a clear, accurate summary. If the snapshot is empty or errors, say so."
    ),
}

router_chain = llm.with_structured_output(AgentRoute)


def router_node(state: State) -> dict:
    decision = router_chain.invoke(
        [SystemMessage(content=ROUTER_PROMPT), *state["messages"]]
    )
    return {"route": decision.target}


def inject_agent_system(state: State) -> dict:
    r = state["route"]
    conv = state["messages"]
    # Summarization flow: navigation agent always runs first with navigator instructions.
    sys_key = "navigator" if r == "summarizer" else r
    sys_text = AGENT_SYSTEM[sys_key]
    return {
        "messages": [
            RemoveMessage(id=REMOVE_ALL_MESSAGES),
            SystemMessage(content=sys_text),
            *conv,
        ]
    }


def general_agent_node(state: State) -> dict:
    return {"messages": [llm.invoke(state["messages"])]}


def route_after_inject(state: State) -> str:
    r = state["route"]
    if r == "summarizer":
        return "navigator_agent"
    return r


def summarizer_prep(state: State) -> dict:
    """Swap navigator system for summarizer system; keep tool + chat history for snapshot."""
    msgs = list(state["messages"])
    body = msgs[1:] if msgs and isinstance(msgs[0], SystemMessage) else msgs
    return {
        "messages": [
            RemoveMessage(id=REMOVE_ALL_MESSAGES),
            SystemMessage(content=AGENT_SYSTEM["summarizer"]),
            *body,
        ]
    }


def after_navigator(state: State):
    if state.get("route") == "summarizer":
        return "summarizer_prep"
    return END


def compile_orchestrator(nav_tools: list, sum_tools: list):
    """Router → inject → general | navigator; summarizer route = navigator then summarizer (snapshot)."""
    navigator_graph = compile_agent_subgraph(nav_tools)
    summarizer_graph = compile_agent_subgraph(sum_tools)
    main = StateGraph(State)
    main.add_node("router", router_node)
    main.add_node("inject_agent_system", inject_agent_system)
    main.add_node("general_agent", general_agent_node)
    main.add_node("navigator_agent", navigator_graph)
    main.add_node("summarizer_prep", summarizer_prep)
    main.add_node("summarizer_agent", summarizer_graph)
    main.add_edge(START, "router")
    main.add_edge("router", "inject_agent_system")
    main.add_conditional_edges(
        "inject_agent_system",
        route_after_inject,
        {
            "general": "general_agent",
            "navigator": "navigator_agent",
            "navigator_agent": "navigator_agent",
        },
    )
    main.add_edge("general_agent", END)
    main.add_conditional_edges(
        "navigator_agent",
        after_navigator,
        {"summarizer_prep": "summarizer_prep", END: END},
    )
    main.add_edge("summarizer_prep", "summarizer_agent")
    main.add_edge("summarizer_agent", END)
    return main.compile()


graph = compile_orchestrator([], [])

In [ ]:
def gradio_history_to_langchain_messages(history: list) -> list:
    '''Map Gradio ChatInterface (type="messages") history to LangChain HumanMessage / AIMessage.'''
    out = []
    for m in history or []:
        role = m.get("role")
        content = m.get("content", "")
        if not isinstance(content, str):
            content = str(content)
        if role == "user":
            out.append(HumanMessage(content=content))
        elif role == "assistant":
            out.append(AIMessage(content=content))
    return out


def _last_assistant_text(messages: list) -> str:
    for m in reversed(messages):
        if isinstance(m, AIMessage):
            c = m.content
            if isinstance(c, str) and c.strip():
                return c.strip()
            if isinstance(c, list):
                parts = [str(x.get("text", x)) for x in c if isinstance(x, dict)]
                if parts:
                    return " ".join(parts).strip()
    return "(No assistant reply.)"


async def respond(message: str, history: list):
    """Router + specialist agents (navigator / summarizer / general) over Playwright MCP."""
    messages = [*gradio_history_to_langchain_messages(history), HumanMessage(content=message)]
    async with MCPServerStdio(
        params=playwright_params, client_session_timeout_seconds=120
    ) as server:
        nav_tools = await build_navigator_tools(server)
        sum_tools = await build_summarizer_tools(server)
        app = compile_orchestrator(nav_tools, sum_tools)
        result = await app.ainvoke(
            {"messages": messages},
            config={"recursion_limit": 35},
        )
    return _last_assistant_text(result["messages"])

### Workflow graph

`graph = compile_orchestrator([], [])` shows topology **without** MCP tools. **Summarizer** routes go **navigator → summarizer_prep → summarizer** (navigation before snapshot). Run **before** `launch()`. Requires graph rendering deps (e.g. `pygraphviz`).

In [ ]:
from IPython.display import Image, display

display(Image(graph.get_graph().draw_mermaid_png()))

In [ ]:
demo = gr.ChatInterface(
    fn=respond,
    type="messages",
    title="LLM chat (LangGraph + Playwright MCP)",
    description="Router → Navigator (navigate/tabs) or Summarizer (snapshot) or General. Playwright MCP via `playwright_params`.",
)

demo.launch(share=False)